In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

class Detector:
    def __init__(self, R, r, N):
        self.R = R
        self.r = r
        self.N = N
        self.positions = self._place_pmt()

    def _place_pmt(self):
        points = []
        while len(points) < self.N:
            p = np.random.uniform(-1, 1, size=3)
            length = np.linalg.norm(p)
            if 0 < length <= 1:
                points.append(p / length * self.R)
        return np.array(points)

def simulate_event_python(E, det, Y=10000, a=0.5):
    N_photons = int(E * Y)
    directions = []
    for _ in range(N_photons):
        phi = np.random.uniform(0, 2 * np.pi)
        cos_theta = np.random.uniform(-1, 1)
        sin_theta = (1 - cos_theta**2)**0.5
        directions.append((sin_theta * np.cos(phi), sin_theta * np.sin(phi), cos_theta))
    
    pmt_list = [tuple(p) for p in det.positions]
    total_electrons = 0
    
    for dx, dy, dz in directions:
        hit_x = dx * det.R
        hit_y = dy * det.R
        hit_z = dz * det.R
        
        for (cx, cy, cz) in pmt_list:
            dist = ((hit_x - cx)**2 + (hit_y - cy)**2 + (hit_z - cz)**2)**0.5
            if dist <= det.r:
                if np.random.uniform(0, 1) < a:
                    total_electrons += 1
    return total_electrons

def simulate_event_vectorized(E, det, Y=10000, a=0.5):
    N_photons = int(E * Y)
    
    phi = np.random.uniform(0, 2 * np.pi, N_photons)
    cos_theta = np.random.uniform(-1, 1, N_photons)
    sin_theta = np.sqrt(1 - cos_theta**2)
    
    x = sin_theta * np.cos(phi)
    y = sin_theta * np.sin(phi)
    z = cos_theta
    
    directions = np.column_stack((x, y, z))
    hit_points = directions * det.R
    
    diffs = hit_points[:, np.newaxis, :] - det.positions[np.newaxis, :, :]
    dists = np.linalg.norm(diffs, axis=2)
    
    hits_mask = dists <= det.r
    num_hits_per_pmt = np.sum(hits_mask, axis=0)
    
    electrons = np.random.poisson(a * num_hits_per_pmt)
    return np.sum(electrons)

if __name__ == "__main__":
    det = Detector(R=2.0, r=0.1, N=40)
    E = 1.0
    
    N_bench = 5
    
    print("Замер времени: чистый Python (5 событий)...")
    start = time.time()
    for _ in range(N_bench):
        simulate_event_python(E, det)
    t_py = time.time() - start
    print(f"Python (цикл по фотонам и ФЭУ): {t_py:.2f} сек")
    
    print("\nЗамер времени: NumPy (100 событий)...")
    start = time.time()
    for _ in range(100):
        simulate_event_vectorized(E, det)
    t_np = time.time() - start
    print(f"NumPy (векторизация): {t_np:.2f} сек на 100 событий")
    
    t_py_per_event = t_py / N_bench
    t_np_per_event = t_np / 100
    print(f"\nВремя на 1 событие:")
    print(f"  Python: {t_py_per_event*1000:.1f} мс")
    print(f"  NumPy:  {t_np_per_event*1000:.3f} мс")
    print(f"  Ускорение: {t_py_per_event / t_np_per_event:.0f}x")
    
    print("\nПроверка корректности (средние значения)...")
    charges_py = [simulate_event_python(E, det) for _ in range(5)]
    charges_np = [simulate_event_vectorized(E, det) for _ in range(500)]
    print(f"Среднее (Python, 5 событий):  {np.mean(charges_py):.1f} ± {np.std(charges_py):.1f}")
    print(f"Среднее (NumPy, 500 событий): {np.mean(charges_np):.1f} ± {np.std(charges_np):.1f}")
    
    print("\nСимуляция 1000 событий для гистограммы...")
    charges_hist = [simulate_event_vectorized(E, det) for _ in range(1000)]
    
    plt.figure(figsize=(8, 4))
    plt.hist(charges_hist, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
    plt.xlabel('Собранный заряд Q')
    plt.ylabel('Количество событий')
    plt.title('Распределение заряда (векторизация, R=2 м, N=40, E=1 МэВ)')
    plt.grid(alpha=0.3)
    plt.show()
    
    print(f"Итоговое среднее: {np.mean(charges_hist):.1f} ± {np.std(charges_hist):.1f}")
    print(f"Относительная ширина: {np.std(charges_hist) / np.mean(charges_hist):.3f}")

Замер времени: чистый Python (5 событий)...
Python (цикл по фотонам и ФЭУ): 1.95 сек

Замер времени: NumPy (100 событий)...
NumPy (векторизация): 1.71 сек на 100 событий

Время на 1 событие:
  Python: 390.4 мс
  NumPy:  17.116 мс
  Ускорение: 23x

Проверка корректности (средние значения)...
